In [1]:
!pip install -q mlflow optuna dagshub


In [2]:
import mlflow
import dagshub

dagshub.init(
    repo_owner="rvaghani",
    repo_name="my-first-repo",
    mlflow=True
)

mlflow.set_experiment("cosmetics_review_classification")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

/Users/ar/Desktop/Riddhi/ESDS_Fall 2025/Pro Data science /Final 
Project/housing_app_fall25/.venv/lib/python3.14/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=b946ac85-453f-4713-8899-92d59f7eb60a&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=ac94bdd6bc3877d9f548fe88fce03c2be22a7e7f1f2d9fdbe26257c35c07d631




Accessing as rvaghani

Initialized MLflow to track repo "rvaghani/my-first-repo"

Repository rvaghani/my-first-repo initialized!

2025/12/17 21:56:37 INFO mlflow.tracking.fluent: Experiment with name 'cosmetics_review_classification' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/67c7b841ce8d4235b1505b346d5cca61', creation_time=1766026597981, experiment_id='0', last_update_time=1766026597981, lifecycle_stage='active', name='cosmetics_review_classification', tags={}>

In [15]:
import pandas as pd

df = pd.read_sql("""
SELECT
    label,
    review_text,
    rating,
    helpful_votes,
    verified_purchase
FROM review
""", engine)

df["verified_purchase"] = df["verified_purchase"].astype(int)

X = df[["review_text", "helpful_votes", "verified_purchase"]]
y = df["label"]


In [16]:
df["verified_purchase"] = df["verified_purchase"].astype(int)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import optuna


/Users/ar/Desktop/Riddhi/ESDS_Fall 2025/Pro Data science /Final Project/housing_app_fall25/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
def run_experiment(model_name, model, use_svd=False, tune=False):
    steps = [
        ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english"))
    ]

    if use_svd:
        steps.append(("svd", TruncatedSVD(n_components=300)))

    steps.append(("clf", model))

    pipe = Pipeline(steps)

    with mlflow.start_run(run_name=f"{model_name}_svd={use_svd}_tune={tune}"):
        pipe.fit(X_train["review_text"], y_train)
        preds = pipe.predict(X_test["review_text"])
        f1 = f1_score(y_test, preds)

        mlflow.log_param("model", model_name)
        mlflow.log_param("svd", use_svd)
        mlflow.log_param("tuned", tune)
        mlflow.log_metric("f1", f1)

        mlflow.sklearn.log_model(pipe, "model")

        print(model_name, "SVD:", use_svd, "Tuned:", tune, "F1:", f1)


In [22]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

models = {
    "logreg": LogisticRegression(max_iter=2000, solver="liblinear"),
    "svm": LinearSVC(),
    "sgd": SGDClassifier(loss="log_loss", random_state=42),
    "rf": RandomForestClassifier(random_state=42, n_jobs=-1)  # ✅ add n_jobs
}


In [19]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score
import mlflow
import joblib, json
from pathlib import Path

Path("../models").mkdir(exist_ok=True)
Path("../metrics").mkdir(exist_ok=True)

def build_pipeline(model, use_svd: bool):
    text_steps = [("tfidf", TfidfVectorizer(max_features=5000, stop_words="english"))]
    if use_svd:
        text_steps.append(("svd", TruncatedSVD(n_components=300, random_state=42)))
    text_pipe = Pipeline(text_steps)

    pre = ColumnTransformer(
        transformers=[
            ("text", text_pipe, "review_text"),
            ("num", "passthrough", ["helpful_votes", "verified_purchase"])
        ]
    )
    return Pipeline([("preprocess", pre), ("clf", model)])


def run_experiment(model_name, model, use_svd=False, tune=False, params=None):
    if params:
        model.set_params(**params)

    pipe = build_pipeline(model, use_svd)

    run_name = f"{model_name}_svd={use_svd}_tune={tune}"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model", model_name)
        mlflow.log_param("svd", use_svd)
        mlflow.log_param("tuned", tune)
        if params:
            for k, v in params.items():
                mlflow.log_param(k, v)

        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        f1 = f1_score(y_test, preds)

        mlflow.log_metric("f1", float(f1))

        # save locally
        model_path = f"../models/{run_name}.joblib"
        metrics_path = f"../metrics/{run_name}.json"

        joblib.dump(pipe, model_path)
        with open(metrics_path, "w") as f:
            json.dump({"f1": float(f1)}, f, indent=2)

        mlflow.log_artifact(model_path, artifact_path="models")
        mlflow.log_artifact(metrics_path, artifact_path="metrics")

        print("✅", run_name, "F1:", f1)
        return f1


In [12]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(X_tr.shape, X_val.shape)


(31920, 4) (7981, 4)


In [24]:
import optuna
from sklearn.base import clone
from sklearn.metrics import f1_score

def tune_params(model_name: str, use_svd: bool, n_trials: int = 3):
    """
    Returns best hyperparams dict for the given model.
    Tunes using (X_tr, y_tr) -> evaluates on (X_val, y_val).
    """
    def objective(trial):
        # create fresh model each trial
        model = clone(models[model_name])

        if model_name == "logreg":
            params = {
                "C": trial.suggest_float("C", 1e-3, 10.0, log=True),
            }
            model.set_params(**params)

        elif model_name == "svm":
            params = {
                "C": trial.suggest_float("C", 1e-3, 10.0, log=True),
            }
            model.set_params(**params)

        elif model_name == "sgd":
            # alpha = regularization strength
            params = {
                "alpha": trial.suggest_float("alpha", 1e-6, 1e-2, log=True),
                "penalty": trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"]),
            }
            model.set_params(**params)

        elif model_name == "rf":
            params = {
               "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=50),
                "max_depth": trial.suggest_int("max_depth", 5, 20, step=5),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 6),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 3),
                "n_jobs": -1
            }
            model.set_params(**params)

        else:
            params = {}

        # build your exact pipeline (same as experiments)
        pipe = build_pipeline(model, use_svd=use_svd)

        pipe.fit(X_tr, y_tr)
        pred = pipe.predict(X_val)
        return f1_score(y_val, pred)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study.best_value


In [25]:
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)


[I 2025-12-17 23:00:09,734] A new study created in memory with name: no-name-1f920b30-1377-4f0d-a5d1-eecf4675fb60


In [28]:
import json
from pathlib import Path
import pandas as pd

rows = []
for p in Path("../metrics").glob("*.json"):
    name = p.stem
    # expect name like: logreg_svd=False_tune=True
    if "_svd=" not in name or "_tune=" not in name:
        continue

    model = name.split("_svd=")[0]
    svd = name.split("_svd=")[1].split("_tune=")[0] == "True"
    tuned = name.split("_tune=")[1] == "True"

    with open(p, "r") as f:
        d = json.load(f)

    rows.append({"model": model, "svd": svd, "tuned": tuned, "f1": d["f1"], "file": p.name})

results_df = pd.DataFrame(rows).sort_values(["model","svd","tuned"]).reset_index(drop=True)
print("Found runs:", len(results_df))
results_df


Found runs: 15


,model,svd,tuned,f1,file
0,logreg,False,False,1.000000,logreg_svd=False_tune=False.json
1,logreg,False,True,1.000000,logreg_svd=False_tune=True.json
2,logreg,True,False,1.000000,logreg_svd=True_tune=False.json
3,logreg,True,True,1.000000,logreg_svd=True_tune=True.json
4,rf,False,False,0.999932,rf_svd=False_tune=False.json
5,rf,False,True,0.998716,rf_svd=False_tune=True.json
6,rf,True,False,1.000000,rf_svd=True_tune=False.json
7,sgd,False,False,0.999865,sgd_svd=False_tune=False.json
8,sgd,False,True,1.000000,sgd_svd=False_tune=True.json
9,sgd,True,False,1.000000,sgd_svd=True_tune=False.json


In [29]:
best_row = results_df.sort_values("f1", ascending=False).iloc[0]
best_row


model                              logreg
svd                                 False
tuned                               False
f1                                    1.0
file     logreg_svd=False_tune=False.json
Name: 0, dtype: object

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from pathlib import Path

# Rebuild the pipeline exactly like before
text_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english"))
])

pre = ColumnTransformer(
    transformers=[
        ("text", text_pipe, "review_text"),
        ("num", "passthrough", ["helpful_votes", "verified_purchase"])
    ]
)

model = LogisticRegression(max_iter=2000, solver="liblinear")

pipe = Pipeline([
    ("preprocess", pre),
    ("clf", model)
])

# Train
pipe.fit(X_train, y_train)

# Save
Path("../models").mkdir(parents=True, exist_ok=True)
joblib.dump(pipe, "../models/best_model.joblib")

print("✅ best_model.joblib saved successfully")


✅ best_model.joblib saved successfully


In [32]:
import joblib
m = joblib.load("../models/best_model.joblib")
m


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers conta